# 01 — Statistical Analysis & Process Drift Exploration
**YieldSense — NAND Test Yield Analytics Pipeline**

This notebook demonstrates exploratory analysis on clean SECOM parametric data:
1. **Point-biserial sensor correlations** against pass/fail bins.
2. **Principal Component Analysis (PCA)** variance decomposition.
3. **Statistical Process Control (SPC)** Shewhart 3-sigma drift monitoring.
4. **Parametric recovery candidate detection** for lot re-binning.

In [ ]:
from src.ingestion.loader import load_from_duckdb
from src.analysis.correlation import compute_sensor_correlations
from src.analysis.pca import perform_pca_analysis
from src.analysis.spc import detect_spc_drift_events
from src.analysis.recovery import identify_recovery_candidates

df = load_from_duckdb('clean_lots')
print(f'Dataset shape: {df.shape[0]} lots, {df.shape[1]} columns')

## 1. Top Failure-Correlated Sensors

In [ ]:
corr_df = compute_sensor_correlations(df)
print('Top 10 failure-correlated sensors:')
print(corr_df.head(10)[['rank', 'sensor', 'correlation', 'p_value', 'is_significant_05']])

## 2. Principal Component Analysis (PCA)

In [ ]:
pca, scaler, summary = perform_pca_analysis(df, variance_threshold=0.95)
print(f'Total sensors: {summary["total_sensors"]}')
print(f'Retained components: {summary["n_components_retained"]}')
print(f'Variance explained: {summary["total_explained_variance"]:.1%}')

## 3. Shewhart 3-Sigma Control Charts (SPC)

In [ ]:
drift_df = detect_spc_drift_events(df, top_n_sensors=5)
print(f'Total 3-sigma drift excursions detected: {len(drift_df)}')
print(drift_df.head(5)[['lot_id', 'timestamp', 'sensor', 'reading_value', 'deviation_sigmas', 'drift_direction']])

## 4. Physical Spec Envelope Recovery Candidates

In [ ]:
candidates_df = identify_recovery_candidates(df, max_oob_sensors=3, margin_std_factor=0.5)
cands = candidates_df['is_recovery_candidate'].sum()
print(f'Recoverable lots: {cands} of {len(candidates_df)} failing units ({cands / len(candidates_df):.1%})')
print(candidates_df[candidates_df['is_recovery_candidate']].head(10))